# Stigma Reduction among Primary Care Clinicians towards People with OUD - A Study and Research Validation Analysis

#### By J M Maxwell - Data Science, Sr. Analyst - CTDS

## Introduction



In this notebook, we’ll be replicating the published analyses by Hooker et al. in A randomized controlled trial of an intervention to reduce stigma toward people with opioid use disorder among primary care clinicians (PMC9922036), in order to demonstrate how users might use these data. These data are available from NIDA Data Share through the HEAL Data Platform.

There is an ongoing, dramatic rise in the number of opioid related overdose deaths in the United States, however only a small portion of patients who are diagnosed with Opioid Use Disorders (OUD) seek treatment and recieve medications for OUD (MOUD). MOUDs, like buprenorphine, can be effective forms of treatment for OUD patients, but only a portion of clinicians are waivered to, and in actuality do, prescribe buprenophine. A potential barrier to care for OUD patients may be the stigma held by primary care clinicians (PCCs) towards people with OUD. There are a limited number of interventions available for reducing OUD stigma among PCCs and a need for analytical support for the efficacy of these interventions. *A randomized controlled trial of an intervention...* examines whether PCCs' stigma towards people with OUD may be reduced and PCCs' intentions to treat OUD patients increased through online training interventions.

This work was done exclusively to demonstrate the advantages of the HEAL Platform's Workspace feature and the ability to utilize data that is joined under the HEAL data mesh. While all the work here was completed by J M. Maxwell and members of the HEAL Platform team, this is not original work, and it is exclusively based off of the work completed by Hooker et al.

The work here does not represent the official opinions, recommendations, or conclusions of Hooker et al. and this work does not represent policy or medical recommendations on behalf of the authors, the NIH HEAL Initiative, The Center For Translational Data Science, or The University of Chicago.



    Hooker, Stephanie A et al. “A randomized controlled trial of an intervention to reduce stigma toward people with opioid use disorder among primary care clinicians.” Addiction science & clinical practice vol. 18,1 10. 11 Feb. 2023, doi:10.1186/s13722-023-00366-1

#### Import Python Libraries

In [1]:
!pip install matplotlib -q
!pip install scikit-learn -q

import zipfile
import pandas as pd
import numpy as np
import scipy
import matplotlib.pyplot as plt
import sklearn
from sklearn import metrics
import os

## Retrieve and validate dataset

In this section we ingest and prepare the data. We're particularly interested in the data fields related to the PCCs' attitude towards people with OUD, their attitudes towards treating people with OUD, and their demographic information. 

In [2]:
if not os.path.exists('stigma_supp.csv'):
    os.system('!gen3 drs-pull object dg.H34L/a6c7fc23-3534-401c-88ca-9fbbb495dcf4')
    with zipfile.ZipFile('ascii-crf-data-files_nida-ctn-0095a2 v1-1.zip', 'r') as zip_ref:
        zip_ref.extractall('.')

In [3]:
!frictionless validate --schema schemas/stigma_supp.json --schema-sync stigma_supp.csv

─────────────────────────────────── Dataset ────────────────────────────────────
                     dataset                      
┏━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ name        ┃ type  ┃ path            ┃ status ┃
┡━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ stigma_supp │ table │ stigma_supp.csv │ VALID  │
└─────────────┴───────┴─────────────────┴────────┘


In [4]:
def clean_df(df):

    col_names = {'DDBS': 'Overall Stigma', 'DDBS_DIFF': 'Difference', 'DDBS_DISDAIN': 'Disdain', 'DDBS_BLAME2': 'Blame', 
              'INTEND_WAIVER': 'Intentions to get waivered', 'INTEND_PRESCRIBE': 'Intentions to prescribe buprenorphine', 
              'WILLWORK': 'Willingness to work with OUD', 'TX_EFFECTIVE': 'Perceived OUD treatment effectiveness', 
              'TX_ADHERENCE': 'Perceived OUD treatment adherence', 'PCC_AGE': 'Age', 'PCC_GENDER': 'Gender', 'PCC_RACE': 'Race', 
              'PCC_HISPANIC': 'Ethnicity', 'STIGMA_GROUP': 'Stigma Group', 'PCC_WAIVERED': 'Waivered to prescribe buprenorphine', 'PCC_MD': 'Degree'}
    df.rename(columns=col_names, inplace=True)
    for field in ['STIG_CLIN_ID', 'HLTH_SYS_ID', 'PCC_ID', 'COMPLETE_TRAINING', 'COMPLETE_SURVEY']:
        if field in df.columns:
            df.drop([field], axis=1, inplace=True)
    
    df.Gender = df.Gender.fillna(7.0)
    df.Ethnicity = df.Ethnicity.fillna(7.0)
    df['Race'] = df.Race.map({'5 White': 'White', '9 Prefer not to answer': 'Prefer not to answer', '2 Asian': 'Asian', '3 Black or African American': 'Black or African American', 
                              '7 Multiple selected': 'Multiple selected', '6 Some other race': 'Some other race'})
    df['Gender'] = df.Gender.map({2.0: 'Female', 1.0: 'Male', 6.0: 'Not Listed', 7.0: 'Prefer not to answer'})
    df['Ethnicity'] = df.Ethnicity.map({0.0: 'Not Hispanic or Latino', 1.0: 'Hispanic or Latino',  7.0: 'Missing'})
    df['Waivered to prescribe buprenorphine'] = df['Waivered to prescribe buprenorphine'].map({0: 'False', 1: 'True'})
    df['Degree'] = df.Degree.map({1: 'MD/DO', 0: 'PA/NP'})

    keep_list = list(set(df.columns) & set(['Stigma Group', 'Overall Stigma', 'Difference', 'Disdain', 'Blame',  
                                                  'Intentions to get waivered', 'Intentions to prescribe buprenorphine', 
                                                  'Willingness to work with OUD', 'Perceived OUD treatment effectiveness', 
                                                  'Perceived OUD treatment adherence', 'Waivered to prescribe buprenorphine', 
                                                  'Gender', 'Ethnicity', 'Race', 'Degree']))
    df = df[keep_list]

    return df

def make_print_df(df):
    
    features = ['Gender', 'Ethnicity', 'Race', 'Waivered to prescribe buprenorphine', 'Degree']
    all_subj = []
    print_df = pd.DataFrame()
    
    N_AC = int(df['Stigma Group'].value_counts()['STIG_CTRL'])
    N_SR = int(df['Stigma Group'].value_counts()['STIG_INT'])
    N = N_AC + N_SR
    
    for feat in features:
        print_df1 = pd.DataFrame({'Category':  f'{feat} - ' + df[feat].value_counts().index, f'All N={N}': (100*df[feat].value_counts()/len(df)).values.round(2)} )
        print_df2 = pd.DataFrame({'Category': f'{feat} - ' + (100*df[df['Stigma Group'] == 'STIG_INT'][feat].value_counts()/len(df[df['Stigma Group'] == 'STIG_INT'])).index, f'Stigma reduction n = {N_SR}': (100*df[df['Stigma Group'] == 'STIG_INT'][feat].value_counts()/len(df[df['Stigma Group'] == 'STIG_INT'])).values.round(2)} )
        print_df3 = pd.DataFrame({'Category':  f'{feat} - ' + (100*df[df['Stigma Group'] == 'STIG_CTRL'][feat].value_counts()/len(df[df['Stigma Group'] == 'STIG_CTRL'])).index, f'Attention-control n = {N_AC}':  (100*df[df['Stigma Group'] == 'STIG_CTRL'][feat].value_counts()/len(df[df['Stigma Group'] == 'STIG_CTRL'])).values.round(2)} )
        
        print_df1 = print_df1.merge(print_df2, how='left', on='Category')
        print_df1 = print_df1.merge(print_df3, how='left', on='Category')
        print_df1.fillna(0, inplace=True)
        print_df1[[f'All N={N}', f'Stigma reduction n = {N_SR}', f'Attention-control n = {N_AC}']] = print_df1[[f'All N={N}', f'Stigma reduction n = {N_SR}', f'Attention-control n = {N_AC}']].astype(str)
        print_df1[f'All N={N}'] = print_df1[f'All N={N}'].apply( lambda x : str(x) + '%')
        print_df1[f'Stigma reduction n = {N_SR}'] = print_df1[f'Stigma reduction n = {N_SR}'].apply( lambda x : str(x) + '%')
        print_df1[f'Attention-control n = {N_AC}'] = print_df1[f'Attention-control n = {N_AC}'].apply( lambda x : str(x) + '%')
        print_df = pd.concat([print_df, print_df1])
        
        if feat != 'Degree':
            print_df = pd.concat([print_df,  pd.DataFrame({'Category': ['-'], f'All N={N}': ['-'], f'Stigma reduction n = {N_SR}': ['-'], f'Attention-control n = {N_AC}': ['-'] })])
            

    return print_df

def make_impact_df(df, features):
    stig_int = []
    stig_ctrl = []
    t_vals = []
    p_vals = []
    d_vals = []
    
    def cohen_d(x,y):
        nx = len(x)
        ny = len(y)
        dof = nx + ny - 2
        return (np.mean(x) - np.mean(y)) / np.sqrt(((nx-1)*np.std(x, ddof=1) ** 2 + (ny-1)*np.std(y, ddof=1) ** 2) / dof)
    
    for feat in features:
        x = df[df['Stigma Group'] == 'STIG_INT'][feat].dropna()
        y = df[df['Stigma Group'] == 'STIG_CTRL'][feat].dropna()
        
        # Perform the t-test
        t_statistic, p_value = scipy.stats.ttest_ind(x, y)
        d = cohen_d(x, y).round(2)
        
        stig_int.append(f'{x.mean().round(1)} ({round(x.std(),1)})')
        stig_ctrl.append(f'{y.mean().round(1)} ({round(y.std(),1)})')
        t_vals.append(t_statistic)
        p_vals.append(p_value)
        d_vals.append(d)
    
    impact_df = pd.DataFrame({"Feature": features, 
                        "Stigma Reduction Mean (SD)": stig_int,
                        "Attention Control Mean (SD)": stig_ctrl,
                        "t-statistic": np.round(t_vals, 3),
                        "p-value": np.round(p_vals, 3),
                        "Cohen's d-value": d_vals})

    return impact_df

In [5]:
df = pd.read_csv('stigma_supp.csv')
df = clean_df(df)
df.head()

,Perceived OUD treatment adherence,Gender,Degree,Difference,Perceived OUD treatment effectiveness,Race,Ethnicity,Stigma Group,Intentions to prescribe buprenorphine,Disdain,Intentions to get waivered,Overall Stigma,Willingness to work with OUD,Waivered to prescribe buprenorphine,Blame
0,2,Male,MD/DO,5.33,2.0,White,Not Hispanic or Latino,STIG_CTRL,3.0,6.33,1.0,5.75,2.00,False,5.5
1,3,Male,MD/DO,3.33,3.0,White,Not Hispanic or Latino,STIG_INT,2.0,5.33,2.0,4.50,3.00,False,5.0
2,2,Male,MD/DO,4.33,3.0,White,Not Hispanic or Latino,STIG_CTRL,3.0,5.33,2.0,4.88,3.00,False,5.0
3,3,Female,MD/DO,3.67,2.0,White,Not Hispanic or Latino,STIG_INT,2.0,4.67,2.0,4.00,2.67,False,3.5
4,2,Female,PA/NP,4.33,3.0,White,Not Hispanic or Latino,STIG_INT,3.0,5.00,3.0,4.25,2.67,False,3.0


## About The Study

The study used by Hooker et. al. for their research was conducted by a nonprofit healthcare organization, HealthPartners, servicing Minnesota and Wisconsin. PCCs who are part of the 15 HealPartners clinics that have access to a CDS tool for identifying and treating people with OUD were invited to participate in the study. The study was a randomized, controlled trial where PCCs recieved one of two trainings: a stigma reduction training or an attention-control training. Following training, the PCCs were assesed at 6 months, 3 months and immediately following completion of the training for their stigma against people with OUD, their intention to get waivered, and their intention to prescribe buprenorphine. Of the 162 PCCs who were eligible, 85 completed both the training and the immediate post training survey (n=46 stigma training and n=39 attention-control training). 

The stigma reduction training was a series of videos recounting OUD patient narratives, examples of using non-stigmatizing language, and explanantions on how to use a Clincal Decision Support (CDS) tool designed to enable PCCs to screen, diagnose, and treat patients with OUD. The attention-control training controlled for contact, attention, and extra CDS tool training, but did not include the OUD patient narratives or examples of non-stigmatized language. 

The outcome measures which were used in this analysis were assessed through a survey immediately following training. The relevant primary and secondary outcome measurements were: 
- *OUD Stigma* - measured using the *Blame*, *Disdain*, and *Difference* scales and the composite *Overall Stigma* scale.
- *Intentions to get waivered or prescribe buprenorphine* - measured on a 1-5 scale the likelihood to get waivered to prescribe buprenorphine or to prescribe buprenorphine should a waiver no longer be required. 
- *Willingness to work with people with OUD* - measured as the average of 3 questionnaire items (each scaled 1-5) from the Drug Problems Perceptions Questionnaire, where a higher score indicates a greater overall willingness to work with patients with OUD.
- *Opioid treatment outcome expectations* - measured on a scale 1-4 on PCCs expecations for the effectiveness and likelihood of patient adherence to OUD treatment.

## Demographic Summary Statistics of PCCs who completed stigma or attention control training 

In [6]:
print_df = make_print_df(df)
print_df

,Category,All N=85,Stigma reduction n = 46,Attention-control n = 39
0,Gender - Female,57.65%,58.7%,56.41%
1,Gender - Male,36.47%,36.96%,35.9%
2,Gender - Prefer not to answer,4.71%,2.17%,7.69%
3,Gender - Not Listed,1.18%,2.17%,0.0%
0,-,-,-,-
0,Ethnicity - Not Hispanic or Latino,94.12%,97.83%,89.74%
1,Ethnicity - Hispanic or Latino,3.53%,2.17%,5.13%
2,Ethnicity - Missing,2.35%,0.0%,5.13%
0,-,-,-,-
0,Race - White,67.06%,63.04%,71.79%


In this first table we have a breakdown of the demographic characteristics of the participating PCCs split amongst the two training formats. There are no significant differences between the stigma and attention-control training groups in their demographics, their medical degree type, or their pre-existing waiver to prescribe burprenorphine. 

## Stigma Reduction and Attention-Control Training on Self-Reported Stigma and Intentions to Treat People 

In [7]:
features = ['Overall Stigma', 'Difference', 'Disdain', 'Blame',  'Intentions to get waivered', 
                'Intentions to prescribe buprenorphine', 'Willingness to work with OUD', 'Perceived OUD treatment effectiveness', 'Perceived OUD treatment adherence']
impact_df = make_impact_df(df, features)
impact_df

,Feature,Stigma Reduction Mean (SD),Attention Control Mean (SD),t-statistic,p-value,Cohen's d-value
0,Overall Stigma,4.1 (1.3),4.2 (1.2),-0.481,0.632,-0.10
1,Difference,3.4 (1.7),3.1 (1.7),0.741,0.461,0.16
2,Disdain,4.7 (1.4),4.9 (1.4),-0.859,0.393,-0.19
3,Blame,4.4 (1.6),4.8 (1.6),-1.286,0.202,-0.28
4,Intentions to get waivered,2.3 (0.7),2.1 (0.8),1.113,0.269,0.26
5,Intentions to prescribe buprenorphine,3.2 (1.0),3.0 (0.9),0.899,0.372,0.21
6,Willingness to work with OUD,3.0 (0.7),3.1 (0.9),-0.828,0.410,-0.18
7,Perceived OUD treatment effectiveness,2.6 (0.8),2.7 (0.7),-0.745,0.459,-0.16
8,Perceived OUD treatment adherence,2.5 (0.6),2.4 (0.6),0.150,0.881,0.03


In the second table we show a number of descriptive statistics for measuring the difference amongst the two training groups in self-reported stigma (Overall Stigma, Difference, Disdain, and Blame), intentions for getting waivered or to prescribe buprenophine if waivered, and perceptions of people with OUD and OUD treatment. The mean and standard deviations for both training groups, as well as, the t-statistic, corresponding p-value, and Cohen's d-value all demonstrate that there are minimal or no significant differences between the two training groups across all measured training outcomes. 

## Correlations among Self-Reported Sitgma and Intentions to Treat People

In [8]:
corr_matrix = df[features].corr().round(2)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
corr_matrix.columns = ['1', '2', '3', '4', '5', '6', '7', '8', '9']
corr_matrix.mask(mask)

,1,2,3,4,5,6,7,8,9
Overall Stigma,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Difference,0.84,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Disdain,0.79,0.47,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Blame,0.64,0.31,0.33,NaN,NaN,NaN,NaN,NaN,NaN
Intentions to get waivered,-0.25,-0.06,-0.23,-0.35,NaN,NaN,NaN,NaN,NaN
Intentions to prescribe buprenorphine,-0.25,-0.11,-0.18,-0.35,0.61,NaN,NaN,NaN,NaN
Willingness to work with OUD,-0.40,-0.34,-0.29,-0.34,0.42,0.46,NaN,NaN,NaN
Perceived OUD treatment effectiveness,-0.32,-0.19,-0.26,-0.38,0.21,0.29,0.46,NaN,NaN
Perceived OUD treatment adherence,-0.39,-0.28,-0.31,-0.35,0.17,0.22,0.36,0.62,NaN


In the final table above, we can see how self-reported stigma, intentions for getting waivered or to prescribe buprenophine if waivered, and perceptions of people with OUD and OUD treatment are related. We use Pearson's correlation coefficients to measure the relation between the measured training outcomes. 

Unsurprisingly, the components for measuring stigma are strongly, positively correlated. We can also see moderate or high positive correlations between Perceived OUD treatment adherence and treatment effectiveness, between PCCs' intentions to get waivered and to prescribe buprenorphine, and between PCCs' willingness to work with OUD patients and the PCCs intentions to prescribe buprenorphine and their perceived OUD treatment effectiveness/adherence. 

There is also a slight or moderate negative correlation between PCCs' self-reported stigma and their intensions to work with OUD patients, their percieved OUD treatment effectiveness/adherence, and their intention to get waivered and prescribe buprenorphine. 

## Conclusions

The study demonstrated there was little relation between the provided stigma reduction training and PCCs' stigma, future intentions, willingness to work with OUD patients, or PCCs' perception of treatment effectiveness or patient treatment adherence. As was expected, there was a relation amongst PCCs' established stigma and their willingness to work with OUD patients and their perceptions of OUD treatment effectiveness and adherence. 

Since greater established PCC stigma is shown to inversely relate to PCCs' willingness to treat people with OUD, intentions to prescribe buprenophine, and perceptions regarding treatment effectiveness and patient adherence to treatment, it is important to find more effective intervention methods for reducing stigma. 

### Citations

    Hooker, Stephanie A et al. “A randomized controlled trial of an intervention to reduce stigma toward people with opioid use disorder among primary care clinicians.” Addiction science & clinical practice vol. 18,1 10. 11 Feb. 2023, doi:10.1186/s13722-023-00366-1

    Hooker, S., Rossum, R., Crain, L., Bart, G. "Reducing Stigma toward People with Opioid Use Disorder among Primary Care Clinicians." NIDA Data Share. May 2024, NIDA-CTN-0095A2


## Combining Multiple Studies

### Read In New Data

In [9]:
synth_df = pd.read_csv('data/synthetic_data.csv')
synth_df.head()

,PCC_ID,PCC_GENDER,PCC_RACE,PCC_HISPANIC,STIGMA_GROUP,INTEND_WAIVER,PCC_WAIVERED,PCC_MD,DDBS,DDBS_DIFF,DDBS_DISDAIN,DDBS_BLAME2
0,2033,1,5 White,0,STIG_INT,2,0,1,6.5,1.7,6.0,5.4
1,2111,2,6 Some other race,0,STIG_CTRL,4,0,0,3.0,5.9,3.4,7.7
2,2031,1,3 Black or African American,0,STIG_CTRL,4,0,0,1.4,6.5,3.9,2.9
3,2156,2,5 White,0,STIG_INT,3,0,0,4.3,4.9,6.4,1.4
4,2160,2,5 White,1,STIG_INT,4,0,1,2.6,6.8,5.8,3.0


### Merge Data Sets

As we can see above, not every field from the corresponding study above is represented in this new dataset. Still we are able to clean the enw data with the same method as before and combine the two studies on their matching fields.

In [10]:
synth_df = clean_df(synth_df)
cols = synth_df.columns
combined_df = pd.concat([df[synth_df.columns], synth_df])
combined_df

,Gender,Degree,Difference,Race,Ethnicity,Stigma Group,Intentions to get waivered,Overall Stigma,Disdain,Waivered to prescribe buprenorphine,Blame
0,Male,MD/DO,5.33,White,Not Hispanic or Latino,STIG_CTRL,1.0,5.75,6.33,False,5.5
1,Male,MD/DO,3.33,White,Not Hispanic or Latino,STIG_INT,2.0,4.50,5.33,False,5.0
2,Male,MD/DO,4.33,White,Not Hispanic or Latino,STIG_CTRL,2.0,4.88,5.33,False,5.0
3,Female,MD/DO,3.67,White,Not Hispanic or Latino,STIG_INT,2.0,4.00,4.67,False,3.5
4,Female,PA/NP,4.33,White,Not Hispanic or Latino,STIG_INT,3.0,4.25,5.00,False,3.0
...,...,...,...,...,...,...,...,...,...,...,...
35,Female,MD/DO,3.30,White,Not Hispanic or Latino,STIG_INT,3.0,5.50,4.60,False,6.2
36,Female,PA/NP,3.80,Black or African American,Not Hispanic or Latino,STIG_CTRL,4.0,6.10,4.10,True,1.2
37,Male,MD/DO,6.60,Black or African American,Not Hispanic or Latino,STIG_INT,1.0,3.40,1.40,False,5.1
38,Female,PA/NP,6.10,Some other race,Not Hispanic or Latino,STIG_INT,2.0,1.50,1.80,False,1.0


### Combined Demographic Summary Statistics

Now, using the newly combined datasets, we are able to recreate some of the work done previously.

In [11]:
combined_print_df = make_print_df(combined_df)
combined_print_df

,Category,All N=125,Stigma reduction n = 63,Attention-control n = 62
0,Gender - Female,56.8%,55.56%,58.06%
1,Gender - Male,39.2%,41.27%,37.1%
2,Gender - Prefer not to answer,3.2%,1.59%,4.84%
3,Gender - Not Listed,0.8%,1.59%,0.0%
0,-,-,-,-
0,Ethnicity - Not Hispanic or Latino,84.0%,87.3%,80.65%
1,Ethnicity - Hispanic or Latino,14.4%,12.7%,16.13%
2,Ethnicity - Missing,1.6%,0.0%,3.23%
0,-,-,-,-
0,Race - White,53.6%,53.97%,53.23%


In [12]:
features = ['Overall Stigma', 'Difference', 'Disdain', 'Blame', 'Intentions to get waivered']
combined_impact_df = make_impact_df(combined_df, features)
combined_impact_df

,Feature,Stigma Reduction Mean (SD),Attention Control Mean (SD),t-statistic,p-value,Cohen's d-value
0,Overall Stigma,4.1 (1.4),4.1 (1.5),0.089,0.930,0.02
1,Difference,3.5 (1.8),3.6 (1.8),-0.188,0.851,-0.03
2,Disdain,4.7 (1.6),4.8 (1.6),-0.340,0.735,-0.06
3,Blame,4.4 (1.8),4.8 (1.9),-1.075,0.284,-0.19
4,Intentions to get waivered,2.3 (0.8),2.6 (1.0),-1.356,0.178,-0.25
